In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
n_loans = 5000

# Basic Loan Details
loan_ids = [f"MSME-{str(i).zfill(5)}" for i in range(1, n_loans + 1)]
sectors = np.random.choice(['Manufacturing', 'Trading', 'Services', 'Agriculture Tech', 'Education', 'Personal Loan', 'Household Loan'], n_loans, p=[0.20, 0.25, 0.15, 0.10, 0.10, 0.10, 0.10])
regions = np.random.choice(['North', 'South', 'East', 'West'], n_loans, p=[0.25, 0.35, 0.15, 0.25])

# Loan Financials
loan_amounts = np.random.uniform(100000, 20000000, n_loans).round(-4)  # Loans b/w 1L and 2cr
tenures = np.random.choice([12, 24, 36, 48, 60], n_loans)
interest_rates = np.random.uniform(11.0, 18.0, n_loans).round(2)

# Dates
start_date = datetime(2021, 1, 1)
end_date = datetime(2026, 7, 19)
days_between = (end_date - start_date).days
disbursement_dates = [start_date + timedelta(days=random.randint(0, days_between)) for _ in range(n_loans)]

# 4. Simulating Risk (DPD - Days Past Due)
# Most loans are good (0 DPD). We inject logic to make Trading/East slightly riskier
dpd_status = []
outstanding_principal = []
for i in range(n_loans):
    # base probability of being late
    risk_score = random.randint(1, 100)

    # Add penalty for specific segments to create a "story" for the dashboard
    if sectors[i] == 'Trading': risk_score += 5
    if regions[i] == 'East': risk_score += 5

    if risk_score <= 85:
        dpd = 0  # paying on time
    elif risk_score <= 92:
        dpd = 15  # slightly late
    elif risk_score <= 96:
        dpd = 45  # High Risk
    elif risk_score <= 98:
        dpd = 75  # Very high Risk
    else:
        dpd = 120  # Extreme high Risk
    
    dpd_status.append(dpd)

    # Calculate EXACT outstanding principal using Loan Amortization Math
    months_active = (datetime(2026, 7, 19) - disbursement_dates[i]).days / 30

    # If the loan is older than its tenure, it should be fully paid (0 outstanding) unless it's an NPA
    if months_active >= tenures[i] and dpd == 0:
        outstanding = 0.0
    else:
        # Cap months active at tenure so we don't get negative outstanding
        k = min(months_active, tenures[i])

        # Monthly interest rate
        r = (interest_rates[i] / 100) / 12
        n = tenures[i]
        P = loan_amounts[i]
        # Amortization Formula for Outstanding Principal after k months:
        # Outstanding = P * [ ((1+r)^n - (1+r)^k) / ((1+r)^n - 1) ]
        if r > 0:
            outstanding = P * (((1 + r)**n - (1 + r)**k) / ((1 + r)**n - 1))
        else:
            outstanding = P - (P / n) * k
    
    outstanding_principal.append(round(outstanding, 2))

# Creating DataFrame and Export
df = pd.DataFrame({
    'Loan_ID': loan_ids,
    'Disbursement_Date': disbursement_dates,
    'Sector': sectors,
    'Region': regions,
    'Loan_Amount': loan_amounts,
    'Tenure_Months': tenures,
    'Interest_Rate': interest_rates,
    'Outstanding_Principal': outstanding_principal,
    'Days_Past_Due': dpd_status
})

df.to_csv('MSME_Loan_Portfolio.csv', index=False)
print("Dataset generated successfully!")
